## XGBoost Model Training and Evaluation

### Imports and Setup

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

import pickle

### Load Originals Data

In [29]:
# Load data from dataset_generators/datasets
final_df_with_lags = pd.read_csv('../../dataset_generators/datasets/final_1_lag_ffa_dataset.csv')

### XGBoost Hyperparameter Tuning

In [30]:
# Function to perform hyperparameter tuning on XGBoost model using GridSearchCV
def hyperparameter_tuning(df):
    # use train_test_split to create a validation set
    train_df, _ = train_test_split(df, test_size=0.2, random_state=42)

    features = train_df.iloc[:, 12:].drop(columns=['fantasy_points'])
    target = train_df['fantasy_points']
    
    scaler = MinMaxScaler()
    features = scaler.fit_transform(features)
    target = scaler.fit_transform(target.values.reshape(-1, 1)).ravel()
    
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.001, 0.01],
        'max_depth': [3, 7],
        'subsample': [0.7, 0.9],
        'colsample_bytree': [0.7, 0.9],
        'gamma': [0, 0.2],
        'reg_alpha': [0, 0.01],
        'reg_lambda': [1, 2]
    }
    
    # Create a KFold cross-validator with shuffling
    cv_shuffled = KFold(n_splits=5, shuffle=True, random_state=42)
    
    scoring = ['neg_root_mean_squared_error', 'r2', 'neg_mean_absolute_error']

    # Initialize XGBRegressor and GridSearchCV using gpu if available
    xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
    grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid,
                               scoring=scoring,
                               refit='neg_root_mean_squared_error',
                               cv=cv_shuffled, verbose=1, n_jobs=-1)
    
    grid_search.fit(features, target)
    print("Best parameters found: ", grid_search.best_params_)
    
    rmse = np.sqrt(-grid_search.best_score_)
    r2 = r2_score(target, grid_search.predict(features))
    mae = mean_absolute_error(target, grid_search.predict(features))
    print("Best RMSE from GridSearchCV: ", rmse)
    print("R² on training data with best estimator: ", r2)
    print("MAE on training data with best estimator: ", mae)
    return grid_search.best_estimator_

# Execute hyperparameter tuning
# xgb_model = hyperparameter_tuning(final_df_with_lags)

In [31]:
# xgb_model

In [32]:
# Save xgb_model using pickle
# with open('models/xgb_model.pkl', 'wb') as f:
#     pickle.dump(xgb_model, f)

### XGBoost Implementation

In [33]:
# Load the models back to verify
with open('models/xgb_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

In [34]:
# Function to load the KFold = 5 normalized data
def load_kfold_data(fold):
    train = pd.read_csv(f'../../dataset_generators/datasets/{fold}_train_normalized.csv')
    test = pd.read_csv(f'../../dataset_generators/datasets/{fold}_test_normalized.csv')
    return train, test

In [35]:
rmse_list = []
r2_list = []
mae_list = []

# Function to perform cross-validation and train XGBoost model
def cross_validate_model():
    for fold in range(5):
        train, test = load_kfold_data(fold)
        X_train = train.iloc[:, 12:].drop(columns=['fantasy_points'])
        y_train = train['fantasy_points']
        X_test = test.iloc[:, 12:].drop(columns=['fantasy_points'])
        y_test = test['fantasy_points']
        # Implement the best estimator from hyperparameter tuning
        model = xgb_model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print("Predictions:", y_pred)
        print("Root Mean Squared Error:", rmse)
        print("R^2 Score:", r2)
        print("Mean Absolute Error:", mae)
        rmse_list.append(rmse)
        r2_list.append(r2)
        mae_list.append(mae)
    
    # Concatenate all test sets
    all_test = pd.concat([load_kfold_data(fold)[1] for fold in range(5)], ignore_index=True)
    # Remove duplicate rows where player_name, season, and week values are the same, keeping the first occurrence
    all_test = all_test.drop_duplicates(subset=['player_name', 'season', 'week'], keep='first')
    test = all_test
    X_test = test.iloc[:, 12:].drop(columns=['fantasy_points'])
    y_test = test['fantasy_points']

    # Concatenate all train sets
    all_train = pd.concat([load_kfold_data(fold)[0] for fold in range(5)], ignore_index=True)
    # Remove duplicate rows where player_name, season, and week values are the same, keeping the first occurrence
    all_train = all_train.drop_duplicates(subset=['player_name', 'season', 'week'], keep='first')
    train = all_train
    X_train = train.iloc[:, 12:].drop(columns=['fantasy_points'])
    y_train = train['fantasy_points']

    print("\n")
    # Calculate mean of RMSE and R^2 Score across all folds
    mean_rmse = sum(rmse_list) / len(rmse_list) if rmse_list else 0
    mean_r2 = sum(r2_list) / len(r2_list) if r2_list else 0
    mean_mae = sum(mae_list) / len(mae_list) if mae_list else 0
    
    # St. Dev. of RMSE and R^2 Score across all folds
    std_rmse = np.std(rmse_list) if rmse_list else 0
    std_r2 = np.std(r2_list) if r2_list else 0
    std_mae = np.std(mae_list) if mae_list else 0
    
    print("Mean RMSE across all folds:", mean_rmse)
    print("Standard Deviation of RMSE across all folds:", std_rmse)
    print("Mean R^2 Score across all folds:", mean_r2)
    print("Standard Deviation of R^2 Score across all folds:", std_r2)
    print("Mean MAE across all folds:", mean_mae)
    print("Standard Deviation of MAE across all folds:", std_mae)
    return model, X_train, y_train, X_test, y_test

# cross_validate_model(final_df_with_lags)
xgb_model, X_train, y_train, X_test, y_test = cross_validate_model()

Predictions: [14.906     13.219347  15.176424  ...  6.6067066  2.6402302  5.237722 ]
Root Mean Squared Error: 5.400781957950902
R^2 Score: 0.4020515346794563
Mean Absolute Error: 3.9654594860046406
Predictions: [ 5.010643   3.3316941 15.915779  ...  2.2524524  3.5249188  3.9120557]
Root Mean Squared Error: 5.447787705734534
R^2 Score: 0.4044958858128155
Mean Absolute Error: 3.9974085631983702
Predictions: [ 6.864746  14.503924  17.173658  ...  3.472192   5.7313037  5.559448 ]
Root Mean Squared Error: 5.482790041418417
R^2 Score: 0.39853787278113084
Mean Absolute Error: 3.9959961037364415
Predictions: [14.684113  16.075705  18.455133  ...  4.5456343  4.006112   5.5963163]
Root Mean Squared Error: 5.345343225942225
R^2 Score: 0.40138543548246364
Mean Absolute Error: 3.923009586375679
Predictions: [ 5.3459377 13.631776  12.570225  ...  5.784899   6.0618     6.6107497]
Root Mean Squared Error: 5.295466184482467
R^2 Score: 0.40656569321874236
Mean Absolute Error: 3.8708248958698714


Mean R

### XGBoost Hyperparameter Tuning with PCA

In [36]:
# Function to perform hyperparameter tuning on XGBoost model with PCA using GridSearchCV
def hyperparameter_tuning(df):
    pipeline = Pipeline([
        ('pca', PCA(n_components=0.95)),  # Retain 95% variance
        ('regressor', XGBRegressor(objective='reg:squarederror', random_state=42))
    ])
    
    # use train_test_split to create a validation set
    train_df, _ = train_test_split(df, test_size=0.2, random_state=42)

    features = train_df.iloc[:, 11:].drop(columns=['fantasy_points'])
    target = train_df['fantasy_points']

    scaler = StandardScaler()
    features = scaler.fit_transform(features)
    target = scaler.fit_transform(target.values.reshape(-1, 1)).ravel()
    
    # Refined parameter grid based on visualizations from later cells
    param_grid = {
        'regressor__n_estimators': [100, 200],
        'regressor__learning_rate': [0.01, 0.02],  # From learning rate plot
        'regressor__max_depth': [3, 7],  # From max_depth plot
        'regressor__subsample': [0.7, 0.9],
        'regressor__colsample_bytree': [0.7, 0.9],
        'regressor__min_child_weight': [1, 3],
        'regressor__reg_alpha': [0.01, 0.1],
        'regressor__reg_lambda': [1, 2]
    }
    
    scoring = ['neg_root_mean_squared_error', 'r2', 'neg_mean_absolute_error']

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        cv=5,
        verbose=1,
        n_jobs=-1,
    )
    
    grid_search.fit(features, target)
    print("Best parameters found: ", grid_search.best_params_)
    
    rmse = np.sqrt(-grid_search.best_score_)
    r2 = r2_score(target, grid_search.predict(features))
    mae = mean_absolute_error(target, grid_search.predict(features))
    pca = grid_search.best_estimator_.named_steps['pca']
    print(f"Explained variance ratio with {pca.n_components_} components: {np.sum(pca.explained_variance_ratio_):.4f}")
    print("Best RMSE from GridSearchCV: ", rmse)
    print("R² on training data with best estimator: ", r2)
    print("MAE on training data with best estimator: ", mae)
    return grid_search.best_estimator_

# Execute hyperparameter tuning
# pca_xgb_model = hyperparameter_tuning(final_df_with_lags)

### XGBoost Model with PCA

In [ ]:
# pca = pca_xgb_model.named_steps['pca']
# print("PCA model parameters:")
# print(pca.get_params())

In [ ]:
# pca_model_save_path = 'models/pca_xgb_model.pkl'
# with open(pca_model_save_path, 'wb') as f:
#     pickle.dump(pca_xgb_model, f)

# # Save best_pca_model using pickle
# with open('../../pca/models/pca_model.pkl', 'wb') as f:
#     pickle.dump(pca, f)

In [38]:
# Load the PCA XGB model back to verify
with open('models/pca_xgb_model.pkl', 'rb') as f:
    pca_xgb_model = pickle.load(f)

In [39]:
# Function to load the KFold PCA data
def load_kfold_pca_data(fold):
    train = pd.read_csv(f'../../dataset_generators/datasets/{fold}_train_pca.csv')
    test = pd.read_csv(f'../../dataset_generators/datasets/{fold}_test_pca.csv')
    return train, test

In [40]:
rmse_list = []
r2_list = []
mae_list = []

# Function to perform 5-fold cross-validation and train XGBoost model with PCA
def cross_validate_model_with_pca():
    for fold in range(5):
        train, test = load_kfold_pca_data(fold)
        X_train_pca = train.iloc[:, 12:].drop(columns=['fantasy_points'])
        y_train = train['fantasy_points']
        X_test_pca = test.iloc[:, 12:].drop(columns=['fantasy_points'])
        y_test = test['fantasy_points']

        # Train XGBoost model with PCA transformed data
        model = pca_xgb_model
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print("Predictions:", y_pred)
        print("Root Mean Squared Error:", rmse)
        print("R^2 Score:", r2)
        print("Mean Absolute Error:", mae)
        rmse_list.append(rmse)
        r2_list.append(r2)
        mae_list.append(mae)
            
    # Concatenate all test sets
    all_test = pd.concat([load_kfold_pca_data(fold)[1] for fold in range(5)], ignore_index=True)
    # Remove duplicate rows where player_name, season, and week values are the same, keeping the first occurrence
    all_test = all_test.drop_duplicates(subset=['player_name', 'season', 'week'], keep='first')
    test = all_test
    X_test_pca = test.iloc[:, 12:].drop(columns=['fantasy_points'])
    y_test = test['fantasy_points']

    # Concatenate all train sets
    all_train = pd.concat([load_kfold_pca_data(fold)[0] for fold in range(5)], ignore_index=True)
    # Remove duplicate rows where player_name, season, and week values are the same, keeping the first occurrence
    all_train = all_train.drop_duplicates(subset=['player_name', 'season', 'week'], keep='first')
    train = all_train
    X_train_pca = train.iloc[:, 12:].drop(columns=['fantasy_points'])
    y_train = train['fantasy_points']
            
    print("\n")
    # Calculate mean of RMSE and R^2 Score across all folds
    mean_rmse = sum(rmse_list) / len(rmse_list) if rmse_list else 0
    mean_r2 = sum(r2_list) / len(r2_list) if r2_list else 0
    mean_mae = sum(mae_list) / len(mae_list) if mae_list else 0
    
    # St. Dev. of RMSE and R^2 Score across all folds
    std_rmse = np.std(rmse_list) if rmse_list else 0
    std_r2 = np.std(r2_list) if r2_list else 0
    std_mae = np.std(mae_list) if mae_list else 0
    
    print("Mean RMSE across all folds:", mean_rmse)
    print("Standard Deviation of RMSE across all folds:", std_rmse)
    print("Mean R^2 Score across all folds:", mean_r2)
    print("Standard Deviation of R^2 Score across all folds:", std_r2)
    print("Mean MAE across all folds:", mean_mae)
    print("Standard Deviation of MAE across all folds:", std_mae)

    return model, X_test_pca, y_test, X_train_pca, y_train

pca_xgb_model, X_test_pca, y_test, X_train_pca, y_train = cross_validate_model_with_pca()

Predictions: [16.949331  13.914644  19.368416  ...  6.2974586  3.6161005  5.0099936]
Root Mean Squared Error: 5.408380674595216
R^2 Score: 0.4003677646752354
Mean Absolute Error: 3.9010589839877587
Predictions: [ 4.7126565  2.4719136 18.618366  ...  1.4654286  3.6104343  3.8261673]
Root Mean Squared Error: 5.439272484679124
R^2 Score: 0.40635604866192565
Mean Absolute Error: 3.924872000571331
Predictions: [ 8.904135  14.157425  19.700132  ...  2.753544   5.7142096  5.3995566]
Root Mean Squared Error: 5.474737581527381
R^2 Score: 0.4003032852914563
Mean Absolute Error: 3.9289324782499624
Predictions: [14.913912  19.671495  17.875235  ...  4.180421   3.3372028  5.0670004]
Root Mean Squared Error: 5.35115103323175
R^2 Score: 0.4000839187798404
Mean Absolute Error: 3.8585837526948645
Predictions: [ 7.4210916 14.434242  14.421357  ...  5.582048   6.756597   7.469759 ]
Root Mean Squared Error: 5.303568029725255
R^2 Score: 0.40474844400891463
Mean Absolute Error: 3.8073558602438267


Mean RMS